# 🎵 YuE2-3B Studio — Google Colab Edition (12 GB VRAM Optimized)
Интерфейс для генерации музыки, MIDI и лупов с выводом в интернет через **ngrok**.

**Инструкция:**
1. В меню сверху выберите: `Среда выполнения (Runtime)` -> `Сменить среду выполнения` -> **T4 GPU**.
2. Укажите ваши токены Hugging Face и Ngrok в форме ниже.
3. Нажмите кнопку `Run` (Выполнить) на ячейках.

In [ ]:
#@title ⚙️ 1. Параметры доступа (HF Token & Ngrok)
import os

HF_TOKEN = "hf_YOUR_HF_TOKEN_HERE" #@param {type:"string"}
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN_HERE" #@param {type:"string"}

os.environ["HF_TOKEN"] = HF_TOKEN.strip()
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN.strip()
print("Токены сохранены в переменных окружения.")

In [ ]:
# 2. Клонирование репозитория и установка зависимостей
!nvidia-smi

%cd /content
!rm -rf YuE2-3B-Studio
!git clone https://github.com/cristie-music/YuE2-3B-Studio.git
%cd /content/YuE2-3B-Studio

# Установка wheel-пакета инференса и системных модулей
!pip install --upgrade pip setuptools wheel
!pip install pyngrok soundfile music21 diffusers==0.40.0 accelerate==1.13.0 transformers==4.49.0 peft==0.14.0 bitsandbytes
!pip install ./yue2_infer-0.1.5-py3-none-any.whl --no-deps

In [ ]:
# 3. Запуск туннеля ngrok и сервера YuE2 Studio
from pyngrok import ngrok
import subprocess
import time

ngrok_token = os.environ.get("NGROK_AUTHTOKEN", "").strip()
if not ngrok_token or ngrok_token == "YOUR_NGROK_AUTHTOKEN_HERE":
    raise ValueError("Пожалуйста, укажите NGROK_AUTHTOKEN в первой ячейке!")

ngrok.set_auth_token(ngrok_token)

# Открываем порт 7860 через ngrok
ngrok_tunnel = ngrok.connect(7860)
public_url = ngrok_tunnel.public_url

print("=" * 65)
print(f"🚀 ВАША ВЕБ-СТУДИЯ ДОСТУПНА ПО ССЫЛКЕ:")
print(f"👉 {public_url}")
print("=" * 65)

# Запуск сервера (выбираем web_server_colab.py, если он существует, иначе web_server.py)
server_script = "web_server_colab.py" if os.path.exists("web_server_colab.py") else "web_server.py"
!python {server_script}